In [1]:
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cpu


In [2]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16 if device=="cuda" else torch.float32)
model = model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [3]:

def generate_with_prompt(system_prompt, user_prompt, max_new_tokens=400, temperature=0.7):
    full_prompt = f"""<|system|>
{system_prompt}</s>
<|user|>
{user_prompt}</s>
<|assistant|>"""

    inputs = tokenizer(full_prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        inputs.input_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only assistant response
    return response.split("<|assistant|>")[-1].strip()

# Test
print(generate_with_prompt(
    system_prompt="You are a knowledgeable and friendly Ethiopian AI assistant.",
    user_prompt="Tell me about the history of Addis Ababa."
))

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Both `max_new_tokens` (=400) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Addis Ababa, also known as Addis Abeba, is the capital and largest city of Ethiopia. It has a rich history spanning over 3,000 years, dating back to the 13th century. The city's origins can be traced back to the 12th century, when it was founded by the Emperor Menelik II.

In the 16th century, Addis Ababa became a major trading center, attracting merchants from across Africa and Europe. In the 19th century, Addis Ababa became the capital of the Ethiopian Empire and was a hub for trade and commerce. During the 20th century, Addis Ababa saw significant development, including the construction of modern buildings and infrastructure, which made it an important hub for trade and business.

Today, Addis Ababa is a vibrant and bustling city, with a diverse population and a rich cultural heritage. The city has several landmarks and attractions, including the Ethiopian Orthodox Tewahedo Church, the Addis Ababa Fistula Hospital, and the National Museum of Ethiopia.

Addis Ababa has also been the 

In [5]:
def ethiopian_ai_assistant(query, mode="normal"):
    if mode == "cot":  # Chain of Thought
        system = "You are an expert Ethiopian culture and knowledge assistant. Think step by step before answering."
    else:
        system = "You are a helpful, accurate, and friendly Ethiopian AI assistant."

    return generate_with_prompt(system, query, temperature=0.65)

# Test different queries
queries = [
    "Recommend 3 traditional Ethiopian foods for a tourist.",

]

for q in queries:
    print(f"\n👤 Question: {q}")
    print("🤖 Answer:", ethiopian_ai_assistant(q)[:500] + "...")

[transformers] Both `max_new_tokens` (=400) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



👤 Question: Recommend 3 traditional Ethiopian foods for a tourist.
🤖 Answer: 1. Tigrinya Fried Rice (Girma Kichma) - This is a classic Ethiopian dish made with rice, onions, garlic, and spices. It is usually served with meat, vegetables, or a sauce made from chickpeas, tomatoes, and onions.

2. Beef Shiro (Lech) - This is a spicy beef stew made with a variety of spices, including cumin, coriander, and turmeric. It is usually served with injera, a flat, thick-wheat flatbread, and vegetables.

3. Beef Tamakgota (Kalandera) - This is a slow-cooked beef stew made with a blen...
